## Silver — Demand Enrichment
Extends `orders_silver` with return flag, time dimensions, and product category join.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

### Read Silver Tables

In [0]:
df_orders   = spark.read.table("databricks_cat.silver.orders_silver")
df_products = spark.read.table("databricks_cat.silver.products_silver")

### Check Products Schema — Detect Category Column

In [0]:
product_cols = df_products.columns
print("products_silver columns:", product_cols)

# Build product lookup — use category if it exists, else default to 'UNKNOWN'
if "category" in product_cols:
    df_prod_lookup = df_products.select(
        "product_id",
        col("product_name") if "product_name" in product_cols else lit(None).cast("string").alias("product_name"),
        upper(trim(col("category"))).alias("category"),
        col("brand") if "brand" in product_cols else lit(None).cast("string").alias("brand")
    )
else:
    df_prod_lookup = df_products.select(
        "product_id",
        col("product_name") if "product_name" in product_cols else lit(None).cast("string").alias("product_name"),
        lit("UNKNOWN").alias("category"),
        col("brand") if "brand" in product_cols else lit(None).cast("string").alias("brand")
    )

df_prod_lookup.printSchema()

### Enrich Orders with Time Dimensions and Return Flag

In [0]:
df_enriched = (
    df_orders
    .withColumn("order_date",   col("order_date").cast("date"))
    .withColumn("year",         year(col("order_date")))
    .withColumn("month",        month(col("order_date")))
    .withColumn("week",         weekofyear(col("order_date")))
    .withColumn("day_of_week",  dayofweek(col("order_date")))
    .withColumn("month_name",   date_format(col("order_date"), "MMMM"))
    # Return flag: negative quantity or zero/negative revenue = return
    .withColumn("is_return",
        when((col("quantity") < 0) | (col("total_amount") <= 0), 1).otherwise(0)
    )
    # Revenue per unit
    .withColumn("unit_price",
        when(col("quantity") != 0, col("total_amount") / col("quantity")).otherwise(lit(0.0))
    )
)

### Join with Product Lookup (Category, Brand)

In [0]:
df_enriched = (
    df_enriched
    .join(df_prod_lookup, on="product_id", how="left")
)
df_enriched.display()

### Write to Silver Delta — orders_demand_silver

In [0]:
(
    df_enriched
    .write.format("delta")
    .mode("overwrite")
    .option("path", "abfss://silver@databricksetestorage.dfs.core.windows.net/orders_demand")
    .saveAsTable("databricks_cat.silver.orders_demand_silver")
)
print("✅ orders_demand_silver written successfully.")

In [0]:
%sql
SELECT * FROM databricks_cat.silver.orders_demand_silver LIMIT 10